<a href="https://colab.research.google.com/github/pngl2415/2.6-openMP/blob/main/Actividad_3_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!nvcc --version

Wed Nov  4 08:34:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [25]:

%%writefile operaciones_arreglos.cu
#include <cuda_runtime.h>
#include <iostream>
#include <iomanip>
#include <cstdlib>
#include <ctime>
#include <climits>
#include <algorithm>
#include <string>

using namespace std;

// ==========================================
// PRACTICA 1.3 - REDUCCIONES COLECTIVAS CUDA
// EQUIPO E01
// ==========================================

// CONFIGURACION DE PRUEBA MASIVA
constexpr int N = 4000000;
constexpr int HILOS_POR_BLOQUE = 256;
constexpr bool MOSTRAR_DETALLES = false;

// Para la prueba de 40 elementos utilizar:
// constexpr int N = 40;
// constexpr int HILOS_POR_BLOQUE = 16;
// constexpr bool MOSTRAR_DETALLES = true;

const char* EQUIPO = "E01";

const char* INTEGRANTES =
    "GUZMAN LOPEZ PAOLA NICOLE, "
    "MONDRAGON GAMBINO JUAN SINUHE, "
    "MORALES LIMON JENNIFER NATALY";

// ==========================================
// VERIFICACION DE ERRORES CUDA
// ==========================================

#define CUDA_CHECK(call) do {                         \
    cudaError_t error = (call);                       \
    if (error != cudaSuccess) {                       \
        cerr << "[HOST] [Equipo: E01] Error CUDA: "    \
             << cudaGetErrorString(error) << endl;    \
        exit(EXIT_FAILURE);                           \
    }                                                 \
} while (0)

// ==========================================
// KERNEL: OPERACIONES ARITMETICAS
// ==========================================

__global__ void kernelAritmetico(
    const int* A,
    const int* B,
    long long* C,
    int n,
    int operacion,
    bool detalle
) {
    int globalIdx =
        blockIdx.x * blockDim.x + threadIdx.x;

    if (globalIdx >= n) return;

    long long resultado = 0;
    const char* nombre = "";

    switch (operacion) {
        case 0:
            resultado = (long long)A[globalIdx]
                      + B[globalIdx];
            nombre = "Suma";
            break;

        case 1:
            resultado = (long long)A[globalIdx]
                      - B[globalIdx];
            nombre = "Resta";
            break;

        case 2:
            resultado = (long long)A[globalIdx]
                      * B[globalIdx];
            nombre = "Multiplicacion";
            break;

        case 3:
            resultado = (long long)A[globalIdx]
                      * A[globalIdx];
            nombre = "Cuadrado";
            break;
    }

    C[globalIdx] = resultado;

    if (detalle) {
        printf(
            "[DEVICE] [Equipo: E01] "
            "[Bloque CUDA: %d] "
            "[Hilo CUDA: %d] "
            "[Indice Global: %d] "
            "[Operacion: %s] -> Valor: %lld\n",
            blockIdx.x,
            threadIdx.x,
            globalIdx,
            nombre,
            resultado
        );
    }
}

// ==========================================
// KERNEL: REDUCCION EN ARBOL BINARIO
// ==========================================
// MODO 0: SUMATORIA
// MODO 1: MAXIMO
// MODO 2: MINIMO
//
// Cada bloque procesa hasta 2*blockDim.x
// elementos usando memoria compartida.
// ==========================================

template <typename T, int MODO>
__global__ void kernelReduccion(
    const int* datos,
    T* parciales,
    int n,
    bool detalle
) {
    // Memoria compartida estatica.
    __shared__ T memoria[HILOS_POR_BLOQUE];

    unsigned int tid = threadIdx.x;

    unsigned int globalIdx =
        blockIdx.x * (blockDim.x * 2) + tid;

    T valor;

    if constexpr (MODO == 0) {
        valor = 0;
    } else if constexpr (MODO == 1) {
        valor = INT_MIN;
    } else {
        valor = INT_MAX;
    }

    if (globalIdx < (unsigned int)n) {
        valor = (T)datos[globalIdx];
    }

    unsigned int segundo =
        globalIdx + blockDim.x;

    if (segundo < (unsigned int)n) {
        T otro = (T)datos[segundo];

        if constexpr (MODO == 0) {
            valor += otro;
        } else if constexpr (MODO == 1) {
            valor = max(valor, otro);
        } else {
            valor = min(valor, otro);
        }
    }

    memoria[tid] = valor;
    __syncthreads();

    if (detalle && globalIdx < (unsigned int)n) {
        printf(
            "[DEVICE] [Equipo: E01] "
            "[Bloque CUDA: %d] "
            "[Hilo CUDA: %d] "
            "[Indice Global: %u] "
            "[Operacion: Reduccion %s] "
            "-> Valor local: %lld\n",
            blockIdx.x,
            tid,
            globalIdx,
            MODO == 0 ? "Sumatoria" :
            MODO == 1 ? "Maximo" : "Minimo",
            (long long)valor
        );
    }

    // REDUCCION EN ARBOL BINARIO
    for (
        unsigned int stride = blockDim.x / 2;
        stride > 0;
        stride >>= 1
    ) {
        if (tid < stride) {
            if constexpr (MODO == 0) {
                memoria[tid] += memoria[tid + stride];
            } else if constexpr (MODO == 1) {
                memoria[tid] = max(
                    memoria[tid],
                    memoria[tid + stride]
                );
            } else {
                memoria[tid] = min(
                    memoria[tid],
                    memoria[tid + stride]
                );
            }
        }

        __syncthreads();
    }

    // El hilo 0 guarda el parcial del bloque.
    if (tid == 0) {
        parciales[blockIdx.x] = memoria[0];

        if (detalle) {
            printf(
                "[DEVICE] [Equipo: E01] "
                "[Bloque CUDA: %d] "
                "[Hilo CUDA: 0] "
                "[Indice Global: %u] "
                "[Operacion: Reduccion %s] "
                "-> Parcial: %lld\n",
                blockIdx.x,
                globalIdx,
                MODO == 0 ? "Sumatoria" :
                MODO == 1 ? "Maximo" : "Minimo",
                (long long)memoria[0]
            );
        }
    }
}

// ==========================================
// CLASE OPERACIONESARREGLOS
// ==========================================

class OperacionesArreglos {
private:
    int* h_A = nullptr;
    int* h_B = nullptr;
    long long* h_C = nullptr;

    int* d_A = nullptr;
    int* d_B = nullptr;
    long long* d_C = nullptr;

    int cantidad = 0;
    bool creados = false;
    bool inicializados = false;

    // TRANSFERENCIA HOST -> DEVICE
    void sincronizarDatos() {
        size_t bytes = cantidad * sizeof(int);

        CUDA_CHECK(cudaMemcpy(
            d_A, h_A, bytes,
            cudaMemcpyHostToDevice
        ));

        CUDA_CHECK(cudaMemcpy(
            d_B, h_B, bytes,
            cudaMemcpyHostToDevice
        ));
    }

    // VERIFICACION DE ARREGLOS
    bool verificar() const {
        if (!creados || !inicializados) {
            cout << "[HOST] [Equipo: E01] "
                 << "Primero cree y llene los arreglos."
                 << endl;
            return false;
        }
        return true;
    }

    // MEDICION DEL TIEMPO GPU
    template <typename F>
    float medirGPU(F ejecutar) {
        cudaEvent_t inicio, fin;

        CUDA_CHECK(cudaEventCreate(&inicio));
        CUDA_CHECK(cudaEventCreate(&fin));

        CUDA_CHECK(cudaEventRecord(inicio));

        ejecutar();

        CUDA_CHECK(cudaGetLastError());

        CUDA_CHECK(cudaEventRecord(fin));
        CUDA_CHECK(cudaEventSynchronize(fin));

        float ms = 0;

        CUDA_CHECK(cudaEventElapsedTime(
            &ms, inicio, fin
        ));

        CUDA_CHECK(cudaEventDestroy(inicio));
        CUDA_CHECK(cudaEventDestroy(fin));

        return ms;
    }

    // OPERACIONES ARITMETICAS
    void operacionAritmetica(
        int tipo,
        const char* nombre
    ) {
        if (!verificar()) return;

        int bloques =
            (cantidad + HILOS_POR_BLOQUE - 1)
            / HILOS_POR_BLOQUE;

        cout << "[HOST] [Equipo: E01] "
             << "Iniciando " << nombre
             << " en GPU." << endl;

        float ms = medirGPU([&]() {
            kernelAritmetico
                <<<bloques, HILOS_POR_BLOQUE>>>(
                    d_A, d_B, d_C,
                    cantidad, tipo,
                    MOSTRAR_DETALLES
                );
        });

        CUDA_CHECK(cudaMemcpy(
            h_C, d_C,
            cantidad * sizeof(long long),
            cudaMemcpyDeviceToHost
        ));

        if (MOSTRAR_DETALLES) {
            for (int i = 0; i < cantidad; i++) {
                cout << "[HOST] [Equipo: E01] "
                     << "[Indice Global: " << i << "] "
                     << nombre
                     << " -> C[" << i << "] = "
                     << h_C[i] << endl;
            }
        } else {
            cout << "[HOST] [Equipo: E01] "
                 << "Muestra C[0] = " << h_C[0]
                 << ", C[N-1] = "
                 << h_C[cantidad - 1] << endl;
        }

        cout << "[HOST] [Equipo: E01] "
             << "Tiempo GPU (" << nombre << "): "
             << fixed << setprecision(4)
             << ms << " ms" << endl;
    }

    // REDUCCIONES COLECTIVAS
    template <typename T, int MODO>
    T ejecutarReduccion(const char* nombre) {
        if (!verificar()) return T{};

        int bloques =
            (cantidad + 2 * HILOS_POR_BLOQUE - 1)
            / (2 * HILOS_POR_BLOQUE);

        T* d_parciales = nullptr;
        T* h_parciales = new T[bloques];

        CUDA_CHECK(cudaMalloc(
            (void**)&d_parciales,
            bloques * sizeof(T)
        ));

        cout << "[HOST] [Equipo: E01] "
             << "Iniciando reduccion " << nombre
             << " en GPU. Bloques: "
             << bloques << endl;

        float ms = medirGPU([&]() {
            kernelReduccion<T, MODO>
                <<<bloques, HILOS_POR_BLOQUE>>>(
                    d_A, d_parciales,
                    cantidad, MOSTRAR_DETALLES
                );
        });

        CUDA_CHECK(cudaMemcpy(
            h_parciales, d_parciales,
            bloques * sizeof(T),
            cudaMemcpyDeviceToHost
        ));

        // CONSOLIDACION FINAL EN HOST
        T resultado;

        if constexpr (MODO == 0) {
            resultado = 0;
        } else if constexpr (MODO == 1) {
            resultado = INT_MIN;
        } else {
            resultado = INT_MAX;
        }

        for (int i = 0; i < bloques; i++) {
            if constexpr (MODO == 0) {
                resultado += h_parciales[i];
            } else if constexpr (MODO == 1) {
                resultado = max(
                    resultado,
                    h_parciales[i]
                );
            } else {
                resultado = min(
                    resultado,
                    h_parciales[i]
                );
            }

            if (MOSTRAR_DETALLES) {
                cout << "[HOST] [Equipo: E01] "
                     << "Consolidando bloque " << i
                     << " -> Parcial: "
                     << h_parciales[i] << endl;
            }
        }

        cout << "[HOST] [Equipo: E01] "
             << "Resultado global (" << nombre
             << "): " << resultado << endl;

        cout << "[HOST] [Equipo: E01] "
             << "Tiempo GPU (" << nombre << "): "
             << fixed << setprecision(4)
             << ms << " ms" << endl;

        CUDA_CHECK(cudaFree(d_parciales));
        delete[] h_parciales;

        return resultado;
    }

public:
    ~OperacionesArreglos() {
        liberarMemoria();
    }

    // LIBERACION DE MEMORIA
    void liberarMemoria() {
        delete[] h_A;
        delete[] h_B;
        delete[] h_C;

        if (d_A) CUDA_CHECK(cudaFree(d_A));
        if (d_B) CUDA_CHECK(cudaFree(d_B));
        if (d_C) CUDA_CHECK(cudaFree(d_C));

        h_A = nullptr;
        h_B = nullptr;
        h_C = nullptr;

        d_A = nullptr;
        d_B = nullptr;
        d_C = nullptr;

        cantidad = 0;
        creados = false;
        inicializados = false;
    }

    // CREAR ARREGLOS DINAMICOS
    void crearArregloCUDA(int n) {
        liberarMemoria();

        cantidad = n;

        h_A = new int[n]{};
        h_B = new int[n]{};
        h_C = new long long[n]{};

        CUDA_CHECK(cudaMalloc(
            (void**)&d_A,
            n * sizeof(int)
        ));

        CUDA_CHECK(cudaMalloc(
            (void**)&d_B,
            n * sizeof(int)
        ));

        CUDA_CHECK(cudaMalloc(
            (void**)&d_C,
            n * sizeof(long long)
        ));

        creados = true;

        cout << "[HOST] [Equipo: E01] "
             << "Arreglos dinamicos A, B y C creados."
             << " N = " << n << endl;
    }

    // LLENADO SECUENCIAL
    void llenarSecuencial() {
        if (!creados) {
            cout << "[HOST] [Equipo: E01] "
                 << "Primero cree los arreglos."
                 << endl;
            return;
        }

        for (int i = 0; i < cantidad; i++) {
            h_A[i] = i + 1;
            h_B[i] = i + 1;
        }

        sincronizarDatos();
        inicializados = true;

        cout << "[HOST] [Equipo: E01] "
             << "Llenado secuencial completado."
             << " A[i] = B[i] = i + 1."
             << endl;
    }

    // LLENADO ALEATORIO
    void llenarAleatorio() {
        if (!creados) {
            cout << "[HOST] [Equipo: E01] "
                 << "Primero cree los arreglos."
                 << endl;
            return;
        }

        for (int i = 0; i < cantidad; i++) {
            h_A[i] = 1 + rand() % 1000000;
            h_B[i] = 1 + rand() % 1000000;
        }

        sincronizarDatos();
        inicializados = true;

        cout << "[HOST] [Equipo: E01] "
             << "Llenado aleatorio completado."
             << " Rango: 1 a 1000000."
             << endl;
    }

    // OPERACIONES ARITMETICAS
    void sumarArreglos() {
        operacionAritmetica(0, "Suma");
    }

    void restarArreglos() {
        operacionAritmetica(1, "Resta");
    }

    void multiplicarArreglos() {
        operacionAritmetica(2, "Multiplicacion");
    }

    void cuadradoArreglo() {
        operacionAritmetica(3, "Cuadrado");
    }

    // REDUCCIONES GPU
    void reduccionSumatoria() {
        ejecutarReduccion<long long, 0>("Sumatoria");
    }

    void reduccionPromedio() {
        if (!verificar()) return;

        long long suma =
            ejecutarReduccion<long long, 0>("Sumatoria");

        double promedio =
            (double)suma / cantidad;

        cout << "[HOST] [Equipo: E01] "
             << "Promedio global: "
             << fixed << setprecision(6)
             << promedio << endl;
    }

    void reduccionMaximo() {
        ejecutarReduccion<int, 1>("Maximo");
    }

    void reduccionMinimo() {
        ejecutarReduccion<int, 2>("Minimo");
    }
};

// ==========================================
// INFORMACION DE LA GPU
// ==========================================

void mostrarGPU() {
    int dispositivo = 0;
    cudaDeviceProp gpu{};

    CUDA_CHECK(cudaGetDevice(&dispositivo));

    CUDA_CHECK(cudaGetDeviceProperties(
        &gpu, dispositivo
    ));

    cout << "[HOST] [Equipo: E01] "
         << "GPU detectada: "
         << gpu.name << endl;

    cout << "[HOST] [Equipo: E01] "
         << "Compute Capability: "
         << gpu.major << "."
         << gpu.minor << endl;

    cout << "[HOST] [Equipo: E01] "
         << "Multiprocesadores: "
         << gpu.multiProcessorCount << endl;

    cout << "[HOST] [Equipo: E01] "
         << "Hilos maximos por bloque: "
         << gpu.maxThreadsPerBlock << endl;

    cout << "[HOST] [Equipo: E01] "
         << "Memoria global: "
         << gpu.totalGlobalMem
         << " bytes" << endl;

    cout << "[HOST] [Equipo: E01] "
         << "Memoria compartida por bloque: "
         << gpu.sharedMemPerBlock
         << " bytes" << endl;
}

// ==========================================
// MENU INTERACTIVO - 12 OPCIONES
// ==========================================

void mostrarMenu() {
    cout << "\n[HOST] [Equipo: E01] "
         << "========== MENU CUDA =========="
         << endl;

    cout << "[HOST] [Equipo: E01] 1. Crear arreglos" << endl;
    cout << "[HOST] [Equipo: E01] 2. Sumar arreglos" << endl;
    cout << "[HOST] [Equipo: E01] 3. Restar arreglos" << endl;
    cout << "[HOST] [Equipo: E01] 4. Multiplicar arreglos" << endl;
    cout << "[HOST] [Equipo: E01] 5. Calcular cuadrado" << endl;
    cout << "[HOST] [Equipo: E01] 6. Llenar secuencial" << endl;
    cout << "[HOST] [Equipo: E01] 7. Llenar aleatorio" << endl;
    cout << "[HOST] [Equipo: E01] 8. Sumatoria GPU" << endl;
    cout << "[HOST] [Equipo: E01] 9. Promedio GPU" << endl;
    cout << "[HOST] [Equipo: E01] 10. Maximo GPU" << endl;
    cout << "[HOST] [Equipo: E01] 11. Minimo GPU" << endl;
    cout << "[HOST] [Equipo: E01] 12. Salir" << endl;

    cout << "[HOST] [Equipo: E01] "
         << "Seleccione opcion: " << flush;
}

// ==========================================
// FUNCION PRINCIPAL
// ==========================================

int main() {
    srand((unsigned int)time(nullptr));

    // PRIMERA LINEA: INTEGRANTES
    cout << "[HOST] [Equipo: " << EQUIPO << "] "
         << "Integrantes: " << INTEGRANTES
         << endl;

    cout << "[HOST] [Equipo: E01] "
         << "Iniciando practica CUDA 1.3"
         << endl;

    mostrarGPU();

    cout << "[HOST] [Equipo: E01] "
         << "Configuracion: N=" << N
         << ", hilos/bloque=" << HILOS_POR_BLOQUE
         << ", detalle=" << MOSTRAR_DETALLES
         << endl;

    OperacionesArreglos programa;
    int opcion = 0;

    do {
        mostrarMenu();

        if (!(cin >> opcion)) {
            cout << "\n[HOST] [Equipo: E01] "
                 << "Fin de entrada." << endl;
            break;
        }

        cout << "\n[HOST] [Equipo: E01] "
             << "Opcion seleccionada: "
             << opcion << endl;

        switch (opcion) {
            case 1:
                programa.crearArregloCUDA(N);
                break;

            case 2:
                programa.sumarArreglos();
                break;

            case 3:
                programa.restarArreglos();
                break;

            case 4:
                programa.multiplicarArreglos();
                break;

            case 5:
                programa.cuadradoArreglo();
                break;

            case 6:
                programa.llenarSecuencial();
                break;

            case 7:
                programa.llenarAleatorio();
                break;

            case 8:
                programa.reduccionSumatoria();
                break;

            case 9:
                programa.reduccionPromedio();
                break;

            case 10:
                programa.reduccionMaximo();
                break;

            case 11:
                programa.reduccionMinimo();
                break;

            case 12:
                cout << "[HOST] [Equipo: E01] "
                     << "Saliendo del programa."
                     << endl;
                break;

            default:
                cout << "[HOST] [Equipo: E01] "
                     << "Opcion no valida."
                     << endl;
        }

    } while (opcion != 12);

    programa.liberarMemoria();

    cout << "[HOST] [Equipo: E01] "
         << "Memoria liberada correctamente."
         << endl;

    // ULTIMA LINEA: INTEGRANTES
    cout << "[HOST] [Equipo: " << EQUIPO << "] "
         << "Integrantes: " << INTEGRANTES
         << endl;

    return 0;
}


Overwriting operaciones_arreglos.cu


In [42]:
!cp log_ejecucion_cuda_masiva.txt log_ejecucion_cuda.txt
!ls -lh log_ejecucion_cuda.txt
!head -n 5 log_ejecucion_cuda.txt
!tail -n 5 log_ejecucion_cuda.txt

-rw-r--r-- 1 root root 5.8K Nov  4 03:50 log_ejecucion_cuda.txt
[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] Iniciando practica CUDA 1.3
[HOST] [Equipo: E01] GPU detectada: Tesla T4
[HOST] [Equipo: E01] Compute Capability: 7.5
[HOST] [Equipo: E01] Multiprocesadores: 40
[HOST] [Equipo: E01] Seleccione opcion: 
[HOST] [Equipo: E01] Opcion seleccionada: 12
[HOST] [Equipo: E01] Saliendo del programa.
[HOST] [Equipo: E01] Memoria liberada correctamente.
[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY


In [41]:
from datetime import datetime
from zoneinfo import ZoneInfo

print("EVIDENCIA 04 - EJECUCION MASIVA CUDA")
print("Equipo: E01")
print("Fecha y hora:", datetime.now(ZoneInfo("America/Mexico_City")).strftime("%d/%m/%Y %H:%M:%S"))

!grep -E 'Configuracion:|Resultado global|Promedio global|Tiempo GPU|Error CUDA' log_ejecucion_cuda_masiva.txt

EVIDENCIA 04 - EJECUCION MASIVA CUDA
Equipo: E01
Fecha y hora: 04/11/2026 03:48:59
[HOST] [Equipo: E01] Configuracion: N=4000000, hilos/bloque=256, detalle=0
[HOST] [Equipo: E01] Resultado global (Sumatoria): 1999722092945
[HOST] [Equipo: E01] Tiempo GPU (Sumatoria): 3.3562 ms
[HOST] [Equipo: E01] Resultado global (Sumatoria): 1999722092945
[HOST] [Equipo: E01] Tiempo GPU (Sumatoria): 0.1717 ms
[HOST] [Equipo: E01] Promedio global: 499930.523236
[HOST] [Equipo: E01] Resultado global (Maximo): 1000000
[HOST] [Equipo: E01] Tiempo GPU (Maximo): 0.1848 ms
[HOST] [Equipo: E01] Resultado global (Minimo): 1
[HOST] [Equipo: E01] Tiempo GPU (Minimo): 0.1829 ms


In [40]:
!printf '1\n7\n8\n9\n10\n11\n12\n' | ./operaciones_arreglos | tee log_ejecucion_cuda_masiva.txt

[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] Iniciando practica CUDA 1.3
[HOST] [Equipo: E01] GPU detectada: Tesla T4
[HOST] [Equipo: E01] Compute Capability: 7.5
[HOST] [Equipo: E01] Multiprocesadores: 40
[HOST] [Equipo: E01] Hilos maximos por bloque: 1024
[HOST] [Equipo: E01] Memoria global: 15637086208 bytes
[HOST] [Equipo: E01] Memoria compartida por bloque: 49152 bytes
[HOST] [Equipo: E01] Configuracion: N=4000000, hilos/bloque=256, detalle=0

[HOST] [Equipo: E01] ========== MENU CUDA ==========
[HOST] [Equipo: E01] 1. Crear arreglos
[HOST] [Equipo: E01] 2. Sumar arreglos
[HOST] [Equipo: E01] 3. Restar arreglos
[HOST] [Equipo: E01] 4. Multiplicar arreglos
[HOST] [Equipo: E01] 5. Calcular cuadrado
[HOST] [Equipo: E01] 6. Llenar secuencial
[HOST] [Equipo: E01] 7. Llenar aleatorio
[HOST] [Equipo: E01] 8. Sumatoria GPU
[HOST] [Equipo: E01] 9. Promedio GPU
[HOST] [Equipo: E01] 10. Maximo G

In [39]:
!nvcc -std=c++17 -O2 operaciones_arreglos.cu -o operaciones_arreglos

In [38]:
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path

archivo = Path("operaciones_arreglos.cu")
codigo = archivo.read_text()

print("EVIDENCIA 03 - CONFIGURACION MASIVA CUDA")
print("Equipo: E01")
print("Fecha y hora:", datetime.now(ZoneInfo("America/Mexico_City")).strftime("%d/%m/%Y %H:%M:%S"))
print("Archivo:", archivo.name)
print("=" * 55)

for numero, linea in enumerate(codigo.splitlines(), 1):
    if linea.strip().startswith((
        "constexpr int N =",
        "constexpr int HILOS_POR_BLOQUE =",
        "constexpr bool MOSTRAR_DETALLES ="
    )):
        print(f"Linea {numero}: {linea}")

EVIDENCIA 03 - CONFIGURACION MASIVA CUDA
Equipo: E01
Fecha y hora: 04/11/2026 03:47:07
Archivo: operaciones_arreglos.cu
Linea 18: constexpr int N = 4000000;
Linea 19: constexpr int HILOS_POR_BLOQUE = 256;
Linea 20: constexpr bool MOSTRAR_DETALLES = false;


In [37]:
!grep -nE '^constexpr (int N|int HILOS_POR_BLOQUE|bool MOSTRAR_DETALLES) =' operaciones_arreglos.cu

18:constexpr int N = 4000000;
19:constexpr int HILOS_POR_BLOQUE = 256;
20:constexpr bool MOSTRAR_DETALLES = false;


In [36]:
!echo "========== INICIO DEL PROGRAMA =========="
!grep 'Integrantes:' log_evidencia_40.txt | head -n 1

!echo ""
!echo "========== FINAL DEL PROGRAMA =========="
!grep 'Integrantes:' log_evidencia_40.txt | tail -n 1

========== INICIO DEL PROGRAMA ==========
[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY

========== FINAL DEL PROGRAMA ==========
[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY


In [35]:
!echo "========== PRIMERA LINEA =========="
!head -n 1 log_evidencia_40.txt

!echo ""
!echo "========== ULTIMA LINEA =========="
!tail -n 1 log_evidencia_40.txt

========== PRIMERA LINEA ==========
[HOST] [Equipo: E01] Fecha y hora: 04/11/2026 09:40:33 UTC

========== ULTIMA LINEA ==========
[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY


In [31]:
!{ date '+[HOST] [Equipo: E01] Fecha y hora: %d/%m/%Y %H:%M:%S %Z'; printf '1\n6\n2\n3\n4\n5\n8\n9\n10\n11\n12\n' | ./operaciones_arreglos_40; } | tee log_evidencia_40.txt

[HOST] [Equipo: E01] Fecha y hora: 04/11/2026 09:40:33 UTC
[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] Iniciando practica CUDA 1.3
[HOST] [Equipo: E01] GPU detectada: Tesla T4
[HOST] [Equipo: E01] Compute Capability: 7.5
[HOST] [Equipo: E01] Multiprocesadores: 40
[HOST] [Equipo: E01] Hilos maximos por bloque: 1024
[HOST] [Equipo: E01] Memoria global: 15637086208 bytes
[HOST] [Equipo: E01] Memoria compartida por bloque: 49152 bytes
[HOST] [Equipo: E01] Configuracion: N=40, hilos/bloque=16, detalle=1

[HOST] [Equipo: E01] ========== MENU CUDA ==========
[HOST] [Equipo: E01] 1. Crear arreglos
[HOST] [Equipo: E01] 2. Sumar arreglos
[HOST] [Equipo: E01] 3. Restar arreglos
[HOST] [Equipo: E01] 4. Multiplicar arreglos
[HOST] [Equipo: E01] 5. Calcular cuadrado
[HOST] [Equipo: E01] 6. Llenar secuencial
[HOST] [Equipo: E01] 7. Llenar aleatorio
[HOST] [Equipo: E01] 8. Sumatoria GPU
[HOST] [Equipo: E

In [34]:
!grep -E 'Fecha y hora:|Integrantes:|Configuracion:|MENU CUDA|1\. Crear arreglos|12\. Salir|\[DEVICE\]|Resultado global|Promedio global|Tiempo GPU' log_evidencia_40.txt | head -n 35

[HOST] [Equipo: E01] Fecha y hora: 04/11/2026 09:40:33 UTC
[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] Configuracion: N=40, hilos/bloque=16, detalle=1
[HOST] [Equipo: E01] ========== MENU CUDA ==========
[HOST] [Equipo: E01] 1. Crear arreglos
[HOST] [Equipo: E01] 12. Salir
[HOST] [Equipo: E01] ========== MENU CUDA ==========
[HOST] [Equipo: E01] 1. Crear arreglos
[HOST] [Equipo: E01] 12. Salir
[HOST] [Equipo: E01] ========== MENU CUDA ==========
[HOST] [Equipo: E01] 1. Crear arreglos
[HOST] [Equipo: E01] 12. Salir
[DEVICE] [Equipo: E01] [Bloque CUDA: 2] [Hilo CUDA: 0] [Indice Global: 32] [Operacion: Suma] -> Valor: 66
[DEVICE] [Equipo: E01] [Bloque CUDA: 2] [Hilo CUDA: 1] [Indice Global: 33] [Operacion: Suma] -> Valor: 68
[DEVICE] [Equipo: E01] [Bloque CUDA: 2] [Hilo CUDA: 2] [Indice Global: 34] [Operacion: Suma] -> Valor: 70
[DEVICE] [Equipo: E01] [Bloque CUDA: 2] [Hilo CUDA: 3] [Indice 

In [33]:
import os
os.environ["TZ"] = "America/Mexico_City"

In [30]:
!nvcc -std=c++17 -O2 operaciones_arreglos_40.cu -o operaciones_arreglos_40

In [29]:
from pathlib import Path

codigo = Path("operaciones_arreglos.cu").read_text()

codigo = codigo.replace(
    "constexpr int N = 4000000;",
    "constexpr int N = 40;"
)

codigo = codigo.replace(
    "constexpr int HILOS_POR_BLOQUE = 256;",
    "constexpr int HILOS_POR_BLOQUE = 16;"
)

codigo = codigo.replace(
    "constexpr bool MOSTRAR_DETALLES = false;",
    "constexpr bool MOSTRAR_DETALLES = true;"
)

Path("operaciones_arreglos_40.cu").write_text(codigo)

print("Archivo operaciones_arreglos_40.cu creado.")

Archivo operaciones_arreglos_40.cu creado.


In [28]:
!grep -E 'Configuracion:|Resultado global|Promedio global|Tiempo GPU|Error CUDA' log_ejecucion_cuda_masiva.txt

[HOST] [Equipo: E01] Configuracion: N=4000000, hilos/bloque=256, detalle=0
[HOST] [Equipo: E01] Resultado global (Sumatoria): 1999711974556
[HOST] [Equipo: E01] Tiempo GPU (Sumatoria): 3.0785 ms
[HOST] [Equipo: E01] Resultado global (Sumatoria): 1999711974556
[HOST] [Equipo: E01] Tiempo GPU (Sumatoria): 0.1549 ms
[HOST] [Equipo: E01] Promedio global: 499927.993639
[HOST] [Equipo: E01] Resultado global (Maximo): 1000000
[HOST] [Equipo: E01] Tiempo GPU (Maximo): 0.1722 ms
[HOST] [Equipo: E01] Resultado global (Minimo): 1
[HOST] [Equipo: E01] Tiempo GPU (Minimo): 0.1700 ms


In [27]:
!printf '1\n7\n8\n9\n10\n11\n12\n' | ./operaciones_arreglos | tee log_ejecucion_cuda_masiva.txt

[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] Iniciando practica CUDA 1.3
[HOST] [Equipo: E01] GPU detectada: Tesla T4
[HOST] [Equipo: E01] Compute Capability: 7.5
[HOST] [Equipo: E01] Multiprocesadores: 40
[HOST] [Equipo: E01] Hilos maximos por bloque: 1024
[HOST] [Equipo: E01] Memoria global: 15637086208 bytes
[HOST] [Equipo: E01] Memoria compartida por bloque: 49152 bytes
[HOST] [Equipo: E01] Configuracion: N=4000000, hilos/bloque=256, detalle=0

[HOST] [Equipo: E01] ========== MENU CUDA ==========
[HOST] [Equipo: E01] 1. Crear arreglos
[HOST] [Equipo: E01] 2. Sumar arreglos
[HOST] [Equipo: E01] 3. Restar arreglos
[HOST] [Equipo: E01] 4. Multiplicar arreglos
[HOST] [Equipo: E01] 5. Calcular cuadrado
[HOST] [Equipo: E01] 6. Llenar secuencial
[HOST] [Equipo: E01] 7. Llenar aleatorio
[HOST] [Equipo: E01] 8. Sumatoria GPU
[HOST] [Equipo: E01] 9. Promedio GPU
[HOST] [Equipo: E01] 10. Maximo G

In [26]:
!nvcc -std=c++17 -O2 operaciones_arreglos.cu -o operaciones_arreglos

In [16]:
!printf '1\n6\n2\n3\n4\n5\n8\n9\n10\n11\n7\n8\n9\n10\n11\n12\n' | ./operaciones_arreglos | tee log_ejecucion_cuda.txt

[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY
[HOST] [Equipo: E01] Iniciando practica CUDA 1.3
[HOST] [Equipo: E01] GPU detectada: Tesla T4
[HOST] [Equipo: E01] Compute Capability: 7.5
[HOST] [Equipo: E01] Multiprocesadores: 40
[HOST] [Equipo: E01] Hilos maximos por bloque: 1024
[HOST] [Equipo: E01] Memoria global: 15637086208 bytes
[HOST] [Equipo: E01] Memoria compartida por bloque: 49152 bytes
[HOST] [Equipo: E01] Configuracion: N=40, hilos/bloque=16, detalle=1

[HOST] [Equipo: E01] ========== MENU CUDA ==========
[HOST] [Equipo: E01] 1. Crear arreglos
[HOST] [Equipo: E01] 2. Sumar arreglos
[HOST] [Equipo: E01] 3. Restar arreglos
[HOST] [Equipo: E01] 4. Multiplicar arreglos
[HOST] [Equipo: E01] 5. Calcular cuadrado
[HOST] [Equipo: E01] 6. Llenar secuencial
[HOST] [Equipo: E01] 7. Llenar aleatorio
[HOST] [Equipo: E01] 8. Sumatoria GPU
[HOST] [Equipo: E01] 9. Promedio GPU
[HOST] [Equipo: E01] 10. Maximo GPU
[HO

In [18]:
!ls -lh log_ejecucion_cuda.txt
!tail -n 15 log_ejecucion_cuda.txt

-rw-r--r-- 1 root root 67K Nov  4 09:15 log_ejecucion_cuda.txt
[HOST] [Equipo: E01] 3. Restar arreglos
[HOST] [Equipo: E01] 4. Multiplicar arreglos
[HOST] [Equipo: E01] 5. Calcular cuadrado
[HOST] [Equipo: E01] 6. Llenar secuencial
[HOST] [Equipo: E01] 7. Llenar aleatorio
[HOST] [Equipo: E01] 8. Sumatoria GPU
[HOST] [Equipo: E01] 9. Promedio GPU
[HOST] [Equipo: E01] 10. Maximo GPU
[HOST] [Equipo: E01] 11. Minimo GPU
[HOST] [Equipo: E01] 12. Salir
[HOST] [Equipo: E01] Seleccione opcion: 
[HOST] [Equipo: E01] Opcion seleccionada: 12
[HOST] [Equipo: E01] Saliendo del programa.
[HOST] [Equipo: E01] Memoria liberada correctamente.
[HOST] [Equipo: E01] Integrantes: GUZMAN LOPEZ PAOLA NICOLE, MONDRAGON GAMBINO JUAN SINUHE, MORALES LIMON JENNIFER NATALY
